<a href="https://colab.research.google.com/github/su-sumico/seminar/blob/main/intro_to_sem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Source: https://www.kaggle.com/code/chansta/introtosem/notebook

* 共分散構造分析（構造方程式モデリング）（SEM: Structural Equation Modeling）をPythonで実行する方法
* How to perform Covariance Structure Analysis (SEM: Structural Equation Modeling) in Python

# Structural Equation Modelling (SEM)

This notebook demonstrates an example of Structural Equation Modelling using [semopy](https://semopy.com/). This [paper](https://www.tandfonline.com/doi/full/10.1080/10705511.2019.1704289?scroll=top&needAccess=true) which provides details on the package is also a useful resource to learn the basic idea of SEM.

This notebook utilise the Griliches dataset on the return of schooling. The main difficulty in that study is the lack of a cohesive measure of ability. It introduces two different measures which *presumed* to have ability as a common factor. This notebook demonstrates one way to capture ability based on the two proxies namely *KWW* and *IQ*, to estimate the impact of ability on learning in addition to the years of schooling.

## Import Modules and Data

Modules required for this notebook are

In [ ]:
!pip install semopy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 15.9 MB/s eta 0:00:00
  Created wheel for semopy: filename=semopy-2.3.10-py3-none-any.whl size=1659681 sha256=7f03da685a486719ed4f2412d61f8cb15a229f8fcf72845d80761a898b8a6e27
  Stored in directory: /root/.cache/pip/wheels/c2/8e/7f/4299ddd66512f11df668a853e9e0814c05da708ebdedb9544f
Successfully built semopy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd '/content/drive/My Drive/sem/'

Mounted at /content/drive
/content/drive/My Drive/sem


In [ ]:
import pandas as pd
import semopy as sem
import graphviz

Next we import the data in the usual fashion.

In [ ]:
# ヘッダありCSV/CSV with headers
data = pd.read_csv("/content/drive/My Drive/sem/griliches.csv", header=0)

Just to confirm the name of each column. Recall some variables have been updated in a subsequent study. Those are the variables with the suffix **variableName80**

In [ ]:
data.columns

Index(['rns', 'rns80', 'mrt', 'mrt80', 'smsa', 'smsa80', 'med', 'iq', 'kww',
       'year', 'age', 'age80', 's', 's80', 'expr', 'expr80', 'tenure',
       'tenure80', 'lw', 'lw80'],
      dtype='object')

In [ ]:
data.head()

,rns,rns80,mrt,mrt80,smsa,smsa80,med,iq,kww,year,age,age80,s,s80,expr,expr80,tenure,tenure80,lw,lw80
0,0,0,0,1,1,1,8,93,35,68,19,31,12,12,0.462,10.635,0,2,5.900,6.645
1,0,0,0,1,1,1,14,119,41,66,23,37,16,18,0.000,11.367,2,16,5.438,6.694
2,0,0,0,1,1,1,14,108,46,67,20,33,14,14,0.423,11.035,1,9,5.710,6.715
3,0,0,0,1,1,1,12,96,32,66,18,32,12,12,0.333,13.089,1,7,5.481,6.477
4,0,0,1,1,1,1,6,74,27,73,26,34,9,11,9.013,14.402,3,5,5.927,6.332


Using semopy normally involves three steps:

    1. Define the model.
    2. Estimate the parameters of the model.
    3. Generate result.

## Define the model

There are three parts:

    1. Measurement equations (relation between the latent variables and their observable measures).
    2. Structural equations. The equations based on theory relating the observed variables with the latent variables as well as other exogenou variables.
    3. Additional covariances in $\pmb{\Theta}$ to be estimated.
    
In the Griliches example, let $A_i$ denotes *ability* then

### Measurement Equations:

\begin{align*}
    iq_i =& \alpha_0 + \alpha_1 A_i + u_{1i} \\
    kww_i =& \gamma_0 + \gamma_1 A_i + \gamma_2 s_i + u_{2i}
\end{align*}

### Structural Equations:

\begin{align*}
    \log w_i =& \beta_0 + \beta_1 A_i + \beta_2 s_i + \beta_3 rns_i \\
            & \quad + \beta_4 mrt_i + \beta_5 smsa_i + \beta_6 med_i + \beta_7 age_i + u_i
\end{align*}

In [ ]:
m1 = """
    #Structural equations

    kww~s+ability
    lw~ability+s+rns+mrt+smsa+med+age

    #Measurement equations - more accurately,
    #it defines which variables are latent and which variables are used to measure the latent variables.

    ability=~iq+kww

    #Additional variances and covariances to be estimated in the Theta matrix.

    iq~~kww
    s~~iq
"""
semM1 = sem.Model(m1)

## Estimate the model

Model defined above can be estimated by using the *fit* method. Several estimators are available. See the manual [here](https://semopy.com/docs/model.html#semopy.model.Model.fit) for more information.

Note: def *fit*
(
self, data=None, cov=None, obj='MLW', solver='SLSQP', groups=None, clean_slate=False, regularization=None, n_samples=None, **kwargs)

In [ ]:
result = semM1.fit(data, obj="GLS")

The object contains various results. Some examples can be found below.

In [ ]:
print(result)

Name of objective: GLS
Optimization method: SLSQP
Optimization successful.
Optimization terminated successfully
Objective value: 0.769
Number of iterations: 21
Params: 0.845 -0.410 0.250 0.047 -0.083 0.109 0.136 0.003 0.050 -0.410 1.139 92.597 8.172 1.641 27.157 0.000


This gives us some basic information about the estimation process. Most importantly is the success completion of the optimization procedure. In order to see the main result, we can call the *inspect* method from the sem object as below:

In [ ]:
print(semM1.inspect())

       lval  op     rval   Estimate  Std. Err    z-value   p-value
0       kww   ~        s   0.845253  0.085037    9.93985       0.0
1       kww   ~  ability  -0.409865  0.088764  -4.617473  0.000004
2        lw   ~  ability   0.250031  0.327213   0.764125  0.444792
3        lw   ~        s   0.047497  0.006212   7.645945       0.0
4        lw   ~      rns  -0.082580  0.026696  -3.093315  0.001979
5        lw   ~      mrt   0.109248  0.026205   4.168965  0.000031
6        lw   ~     smsa   0.136235  0.025876   5.264827       0.0
7        lw   ~      med   0.003198  0.004545   0.703583  0.481693
8        lw   ~      age   0.049538  0.004885  10.140278       0.0
9        iq   ~  ability   1.000000         -          -         -
10      kww   ~  ability  -0.409865  0.088764  -4.617473  0.000004
11  ability  ~~  ability   1.641200  2.178557   0.753343  0.451244
12       iq  ~~      kww   1.139403  1.910046   0.596532   0.55082
13       iq  ~~       iq  92.597055  2.618286  35.365522      

## Generate (Pretty) Results

One strength about this module is its ability to generate results in a report format. It can also generate the graphical representation of the SEM.

To generate the graphical representation of your system use

In [ ]:
g = sem.semplot(m1,'griliches.png')

To generate an HTML report on your result use:

In [ ]:
folderName = "/content/drive/My Drive/sem/GrilichesResultDemo"
sem.report(semM1, folderName)

## Kaggle

If you are using this on Kaggle, then the code below allows you to zip the report generate above. You can then download the zip file for further inspection.

In [ ]:
import os as os
savefilename = folderName+'.zip'
os.system('zip -r {0} {1}'.format(savefilename, folderName))

3072

## Another Example

In this section, we will visit the Bollen (1989) example on democracy. This is part of the examples file within [semopy](http://semopy.com) so we can extract the model specification and the data directly from the module.

Observed variables:

- $y_1$: freedom of the press, 1960
- $y_2$: freedom of political opposition, 1960
- $y_3$: fairness of elections, 1960
- $y_4$: effectivness of elected legislature, 1960
- $y_5$ - $y_8$ are the same variables as above, respectively, measured in 1965
- $x_1$: GNP per capita, 1960
- $x_2$: energy consumption per capita, 1960
- $x_3$: percentage of labor force in industry, 1960

Latent Variables:

- $ind60$: exogenous latent variable on industralization.
- $dem60$: endogenous latent variable on democracy at 1960.
- $dem65$: endogenou latent variable on democracy at 1965.
	    


* 構造方程式モデリング(SEM)semopyのチュートリアル/Structural Equation Modeling (SEM) semopy tutorial
 * https://qiita.com/roki18d/items/6d923d4319e2048885fa
 * https://qiita.com/h-fkn/items/4a44559748e0ef4a2c4a

Political Democracy: https://semopy.com/report_example/report.html

In [ ]:
from semopy.examples import political_democracy # loading up the submodules

desc = political_democracy.get_model() # model has been defined already.
data = political_democracy.get_data() # getting the data from the module.
print(desc) # print the model specification. You can change

# measurement model
ind60 =~ x1 + x2 + x3
dem60 =~ y1 + y2 + y3 + y4
dem65 =~ y5 + y6 + y7 + y8
# regressions
dem60 ~ ind60
dem65 ~ ind60 + dem60
# residual correlations
y1 ~~ y5
y2 ~~ y4 + y6
y3 ~~ y7
y4 ~~ y8
y6 ~~ y8


* inspect() メソッドで、パラメータ推定結果を確認できます。
* The inspect() method allows you to check the results of parameter estimation.

In [ ]:
mod = sem.Model(desc)
res_opt = mod.fit(data)
estimates = mod.inspect()
estimates

,lval,op,rval,Estimate,Std. Err,z-value,p-value
0,dem60,~,ind60,1.482379,0.399024,3.715017,0.000203
1,dem65,~,ind60,0.571912,0.221383,2.583364,0.009784
2,dem65,~,dem60,0.837574,0.098446,8.507992,0.0
3,x1,~,ind60,1.000000,-,-,-
4,x2,~,ind60,2.180494,0.138565,15.736254,0.0
5,x3,~,ind60,1.818546,0.151993,11.96465,0.0
6,y1,~,dem60,1.000000,-,-,-
7,y2,~,dem60,1.256819,0.182687,6.879647,0.0
8,y3,~,dem60,1.058174,0.151521,6.983699,0.0
9,y4,~,dem60,1.265186,0.145151,8.716344,0.0


Generating the graphical representation of the model and the result folder.

In [ ]:
folderName='/content/drive/My Drive/sem/DemocracyResult'
gmod = sem.semplot(mod,'democracy.png')
sem.report(mod, folderName)

Again, if you are doing this on Kaggle, we want to be able to download the report easily.

In [ ]:
savefilename = folderName+'.zip'
os.system('zip -r {0} {1}'.format(savefilename, folderName))

3072

* 参考/Reference: https://note.com/k_fukunaka/n/n994613904942
* 参考/Reference: https://toukei-lab.com/covariance-structure-analysis#google_vignette